# Thesis — exact-case A1–A5 qualitative heatmap comparisons

**Inspection-first, no inference or heatmap extraction.** Reproduces the fusion notebook's case order, overlay function, box appearance, four-row pagination, fonts, and PDF export. Requires *real* A1–A5 directories and actual field paths to be supplied **after inspecting the saved payloads**. Fails closed on missing maps or inconsistent labels. It does not compute localization metrics.

**Important provenance:** the uploaded reference notebook provides the canonical BioViL manifest path, not the manifest's contents. The 50 IDs cannot be recovered offline without that manifest or the previously exported `canonical_100_comparison_gallery_order.csv`. This notebook consumes either exact existing artifact, never creates a new sample.

**Output:** three requested PDFs, `canonical_50_image_selection_manifest.csv`, `canonical_100_image_pathology_manifest.csv`, `coverage_report.csv`, `coverage_summary.json`, and an optional compact thesis figure.


In [ ]:
from pathlib import Path
import json, math, collections, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

TARGET_LABELS = ['Cardiomegaly', 'Pleural effusion']
CASES_PER_PAGE, FIG_DPI = 4, 150
CMAP_NAME, OVERLAY_ALPHA = 'magma', 0.50
OUTPUT_DIR = Path('./anatomy_heatmap_comparisons')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Configure **existing** input paths

The original ALBEF and VinDr paths below come directly from the reference notebook and may require updating on another machine. **No A1–A5 paths were supplied**: replace each `None` with the actual existing output folder. One field selector is required for each distinct representation; leave those as `None` until inspecting real files in §3.


In [ ]:
ORIGINAL_DIR = Path('/home/vault/iwi5/iwi5362h/outputs/original_albef_itc_margin_gradcam_test_best_cardio')
CANONICAL_GALLERY_CSV = Path('/home/vault/iwi5/iwi5362h/results/visualization/original_fc_gated_mean_residual_transformer_heatmaps/canonical_100_comparison_gallery_order.csv')
BIOVIL_MANIFEST_CSV = Path('/home/vault/iwi5/iwi5362h/results/biovil_t_phrase_grounding/same_50_albef_margin/manifest.csv')
IMAGES_ROOT = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/test')
ANNOTATIONS_CSV = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/annotations/annotations_test.csv')
IMAGE_METADATA_CSV = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/test_meta.csv')

# Set these to real directories. A5 raw/learned can share a directory if both live in one payload.
SOURCE_DIRS = {
    'original': ORIGINAL_DIR,
    'A1': None,
    'A2': None,
    'A3_001': None,
    'A4': None,
    'A3_0005': None,
    'A3_0001': None,
    'A5_raw': None,
    'A5_head': None,
}

# Dot-separated paths inside each pathology record. Examples only, NOT inferred schemas.
# Original ALBEF field below is VERIFIED FROM its extraction script and reference notebook.
MAP_FIELD_PATHS = {
    'original': 'cam_vis_up',
    'A1': None, 'A2': None, 'A3_001': None, 'A4': None,
    'A3_0005': None, 'A3_0001': None,
    'A5_raw': None, 'A5_head': None,
}

# Some extractors save visualizations at native patch resolution or in other ranges.
# Each source needs confirmation from actual payloads before plotting. Choose 'stored_01'
# only if the selected stored field is a DISPLAY-READY [0,1] map.
DISPLAY_CONVENTION = {'original':'stored_01', **{k: None for k in SOURCE_DIRS if k != 'original'}}

# Configured *only after inspection* if the corresponding stored field has known
# fixed value boundaries; use 'fixed_range' and give (min,max) in DISPLAY_RANGES.
DISPLAY_RANGES = {k: None for k in SOURCE_DIRS}

PDF_SPECS = [
    ('compare_original_A1_A2_A3_A4_heatmaps.pdf',
     [('original','Original ALBEF'),('A1','A1 - Direct support'),('A2','A2 - Uncertainty-weighted'),
      ('A3_001','A3 - lambda=0.01'),('A4','A4 - Classification preservation (1)')]),
    ('compare_original_A3_lambda_ablation_heatmaps.pdf',
     [('original','Original ALBEF'),('A3_001','A3 - lambda=0.01'),
      ('A3_0005','A3.1 - lambda=0.005'),('A3_0001','A3.2 - lambda=0.001')]),
    ('compare_original_A3_best_A5_heatmaps.pdf',
     [('original','Original ALBEF'),('A3_0001','A3.2 - lambda=0.001'),
      ('A5_raw','A5 - Raw pathology cross-attention'),('A5_head','A5 - Learned patch-head output')]),
]


## 2. Recover exact canonical pair order (no new selection)

Prefer the gallery CSV written by the reference notebook. If absent, reconstruct only from the original canonical BioViL-T manifest using its stable pathology ordering, exactly as the reference does. The notebook verifies that all 50 unique IDs occur once per target pathology, that pairs are unique, and that the first 50 rows are Cardiomegaly.


In [ ]:
def normal_label(x): return str(x).strip().casefold()

def exact_canonical_pairs():
    if CANONICAL_GALLERY_CSV.is_file():
        raw = pd.read_csv(CANONICAL_GALLERY_CSV, dtype={'image_id':str})
        provenance = str(CANONICAL_GALLERY_CSV)
    elif BIOVIL_MANIFEST_CSV.is_file():
        raw = pd.read_csv(BIOVIL_MANIFEST_CSV, dtype={'image_id':str})
        if not {'image_id','label','heatmap_path'}.issubset(raw.columns):
            raise KeyError('Canonical BioViL manifest must have image_id, label, heatmap_path')
        raw = raw[raw.label.map(normal_label).isin(map(normal_label,TARGET_LABELS))].copy()
        raw['_sort_label'] = raw.label.map({normal_label(s):i for i,s in enumerate(TARGET_LABELS)})
        raw = raw.sort_values('_sort_label',kind='stable')
        provenance = str(BIOVIL_MANIFEST_CSV)
    else:
        raise FileNotFoundError('Need EXISTING reference gallery CSV or BioViL-T canonical manifest; cannot reconstruct exact IDs without one.')
    if not {'image_id','label'}.issubset(raw.columns): raise KeyError('Canonical file requires image_id and label')
    raw = raw[['image_id','label']].copy().reset_index(drop=True)
    raw['image_id'] = raw.image_id.astype(str)
    raw['label'] = raw.label.map(lambda s: next((x for x in TARGET_LABELS if normal_label(x)==normal_label(s)),s))
    if raw.duplicated(['image_id','label']).any(): raise ValueError('Duplicate canonical image/pathology pairs')
    assert len(raw)==100, f'Expected 100 canonical rows, got {len(raw)}'
    assert raw.image_id.nunique()==50, f'Expected 50 unique image IDs, got {raw.image_id.nunique()}'
    expected = [(p, i) for p in TARGET_LABELS for i in raw.iloc[:50].image_id.tolist()]
    actual = list(zip(raw.label,raw.image_id))
    if actual != expected:
        raise ValueError('Canonical pair structure/order differs: expected SAME 50 IDs in SAME order for Cardiomegaly then Pleural effusion')
    return raw, provenance

pairs, case_source = exact_canonical_pairs()
unique_ids = pairs.iloc[:50].image_id.tolist()
pd.DataFrame({'selection_order':range(1,51),'image_id':unique_ids}).to_csv(OUTPUT_DIR/'canonical_50_image_selection_manifest.csv',index=False)
pairs.assign(pair_order=range(1,101),selection_source=case_source).to_csv(OUTPUT_DIR/'canonical_100_image_pathology_manifest.csv',index=False)
print('Case provenance:', case_source)
display(pairs.head(6))


## 3. Inspect saved outputs **before** selecting fields

This inspection is intentionally schema-neutral. It reports top-level keys, nested key paths, shape, dtype, range, available `ground_truth`, embedded `image_id`/`label`, and model/extractor metadata. It does **not** produce/modify heatmaps. Review its output before setting `MAP_FIELD_PATHS` and `DISPLAY_CONVENTION` in §1 and re-running from the top.


In [ ]:
def load_pt(path):
    try: return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError: return torch.load(path, map_location='cpu')

def inspect_tree(obj, prefix='', depth=0):
    if depth>4: return []
    if isinstance(obj,dict):
        out=[]
        for k,v in obj.items(): out.extend(inspect_tree(v, f'{prefix}.{k}' if prefix else str(k),depth+1))
        return out
    if torch.is_tensor(obj) or isinstance(obj,np.ndarray):
        a=np.asarray(obj.detach().cpu() if torch.is_tensor(obj) else obj)
        mini,maxi=(float(a.min()),float(a.max())) if a.size and np.issubdtype(a.dtype,np.number) else (None,None)
        return [dict(field=prefix,shape=tuple(a.shape),dtype=str(a.dtype),minimum=mini,maximum=maxi)]
    if isinstance(obj,(str,int,float,bool)) or obj is None:
        return [dict(field=prefix,value=str(obj)[:170])]
    return [dict(field=prefix,type=str(type(obj)))]

def list_pt_files(root):
    root=Path(root)
    if not root.is_dir(): raise FileNotFoundError(root)
    paths=list(root.rglob('*.pt'))
    if not paths: raise FileNotFoundError(f'No .pt files in {root}')
    return sorted(paths)

inspection=[]
for name, directory in SOURCE_DIRS.items():
    if directory is None:
        print(f'{name}: UNCONFIGURED — supply existing output directory')
        continue
    paths=list_pt_files(directory)
    sample=next((p for p in paths if p.stem==unique_ids[0]), paths[0])
    payload=load_pt(sample)
    print(f'\n{name}: directory={directory} files={len(paths)} example={sample}')
    print('ROOT KEYS:', list(payload) if isinstance(payload,dict) else type(payload))
    tree=inspect_tree(payload)
    display(pd.DataFrame(tree).head(80))
    inspection.append(dict(source=name,directory=str(directory),file_count=len(paths),sample=str(sample),fields=tree))
with open(OUTPUT_DIR/'payload_inspection.json','w') as f: json.dump(inspection,f,indent=2)
print('Saved inspection to:',OUTPUT_DIR/'payload_inspection.json')


## 4. Build ID-index and validate maps and GT across models

No implicit fallback keys. Exactly one `.pt` file per (ID, pathology) is required, either `{image_id}.pt` containing nested per-label records or `{image_id}__{label_slug}.pt` with explicit label metadata. Duplicate matches are rejected. If your actual naming convention differs, adapt `index_files()` transparently after inspection. For label-specific files without embedded identity/label metadata, indexing stops rather than guessing.


In [ ]:
def slug(label): return normal_label(label).replace(' ','_')

def index_files(source, directory):
    paths=list_pt_files(directory); index={}; ambiguous=[]
    for path in paths:
        stem=path.stem
        for image_id in unique_ids:
            if stem==image_id:
                for label in TARGET_LABELS:
                    key=(image_id,label)
                    if key in index: ambiguous.append((key,str(index[key]),str(path)))
                    else: index[key]=path
                break
            for label in TARGET_LABELS:
                if stem==f'{image_id}__{slug(label)}':
                    key=(image_id,label)
                    if key in index: ambiguous.append((key,str(index[key]),str(path)))
                    else: index[key]=path
    if ambiguous: raise ValueError(f'{source}: duplicate/ambiguous files: {ambiguous[:10]}')
    return index

def find_path(obj, field):
    value=obj
    for part in field.split('.'):
        if not isinstance(value,dict) or part not in value: raise KeyError(f'Field {field!r} missing at {part!r}; keys={list(value) if isinstance(value,dict) else type(value)}')
        value=value[part]
    return value

def extract_record(payload, image_id,label,path):
    if not isinstance(payload,dict): raise TypeError(f'{path}: expected dict payload')
    meta=payload.get('__metadata__',{})
    if not isinstance(meta,dict): meta={}
    embedded_id=meta.get('image_id',payload.get('image_id'))
    if embedded_id is not None and str(embedded_id)!=image_id:
        raise ValueError(f'{path}: embedded image ID {embedded_id} != {image_id}')
    matching=[k for k in payload if normal_label(k)==normal_label(label)]
    if len(matching)>1: raise ValueError(f'{path}: duplicate pathology keys for {label}')
    if len(matching)==1:
        rec=payload[matching[0]]
    else:
        embedded_label=payload.get('label',meta.get('label'))
        if embedded_label is None or normal_label(embedded_label)!=normal_label(label):
            raise KeyError(f'{path}: no explicit matching pathology record/label for {label}')
        rec=payload
    if not isinstance(rec,dict): raise TypeError(f'{path}/{label}: record is not dict')
    rec_id=rec.get('image_id')
    if rec_id is not None and str(rec_id)!=image_id: raise ValueError(f'{path}: record ID mismatches')
    rec_label=rec.get('label')
    if rec_label is not None and normal_label(rec_label)!=normal_label(label): raise ValueError(f'{path}: record pathology mismatch')
    return rec,meta

def check_map(source, record, img_size):
    field=MAP_FIELD_PATHS[source]
    if not field: raise ValueError(f'{source}: MAP_FIELD_PATHS not configured after inspection')
    arr=find_path(record,field)
    arr=np.asarray(arr.detach().cpu().float() if torch.is_tensor(arr) else arr,dtype=np.float32).squeeze()
    if arr.ndim!=2 or min(arr.shape)<2: raise ValueError(f'{source}: incompatible 2D shape {arr.shape}')
    if not np.isfinite(arr).all(): raise ValueError(f'{source}: map contains NaN/Inf')
    conv=DISPLAY_CONVENTION[source]
    if conv=='stored_01':
        if arr.min()<-1e-5 or arr.max()>1+1e-5: raise ValueError(f'{source}: stored_01 specified but range [{arr.min()},{arr.max()}]')
    elif conv=='fixed_range':
        limits=DISPLAY_RANGES[source]
        if not limits or len(limits)!=2 or limits[1]<=limits[0]: raise ValueError(f'{source}: invalid fixed range')
        arr=(arr-float(limits[0]))/(float(limits[1])-float(limits[0]))
    else: raise ValueError(f'{source}: unknown/unconfirmed display convention: {conv}')
    # Explicit spatial resizing is allowed solely to align a saved map with the original radiograph.
    if arr.shape != (img_size[1],img_size[0]):
        from PIL import Image as PILImage
        arr=np.asarray(PILImage.fromarray(arr,mode='F').resize(img_size,resample=PILImage.Resampling.BILINEAR),dtype=np.float32)
        transformed=True
    else: transformed=False
    return np.clip(arr,0,1),transformed

required_sources=list(SOURCE_DIRS)
unconfigured=[s for s in required_sources if SOURCE_DIRS[s] is None or MAP_FIELD_PATHS[s] is None or DISPLAY_CONVENTION[s] is None]
if unconfigured: raise RuntimeError('Inspection gate: configure directory, actual field and normalization for: '+', '.join(unconfigured))
indices={s:index_files(s,d) for s,d in SOURCE_DIRS.items()}
if not IMAGES_ROOT.is_dir(): raise FileNotFoundError(IMAGES_ROOT)
records={}; issues=[]; rows=[]; transformations=[]
for row in pairs.itertuples(index=False):
    image_id,label=row.image_id,row.label
    img_path=IMAGES_ROOT/f'{image_id}.png'
    try:
        with Image.open(img_path) as im: size=im.size
    except Exception as e:
        size=None; issues.append(f'IMAGE {image_id}: {e}')
    gt_values=[]
    for source in required_sources:
        key=(image_id,label); pt=indices[source].get(key)
        status='ok'; detail=''; shape=''; gt=np.nan
        try:
            if pt is None: raise FileNotFoundError(f'No exact file match for {key}')
            if size is None: raise FileNotFoundError(f'Image unavailable {img_path}')
            rec,meta=extract_record(load_pt(pt),image_id,label,pt)
            raw=find_path(rec,MAP_FIELD_PATHS[source]); raw=np.asarray(raw.detach().cpu() if torch.is_tensor(raw) else raw).squeeze()
            shape=str(tuple(raw.shape)); vis,resized=check_map(source,rec,size)
            gt=float(rec.get('ground_truth',np.nan))
            if np.isfinite(gt): gt_values.append((source,gt))
            records[(source,image_id,label)]={'map':vis,'record':rec,'metadata':meta,'path':str(pt)}
            transformations.append(dict(source=source,image_id=image_id,label=label,source_shape=shape,display_shape=str(vis.shape),bilinear_resize=resized,display_convention=DISPLAY_CONVENTION[source],field=MAP_FIELD_PATHS[source]))
        except Exception as e: status='error';detail=f'{type(e).__name__}: {e}';issues.append(f'{source}/{image_id}/{label}: {detail}')
        rows.append(dict(image_id=image_id,label=label,source=source,path=str(pt) if pt else '',status=status,detail=detail,shape=shape,ground_truth=gt))
    if len({v for _,v in gt_values})>1: issues.append(f'GT mismatch {image_id}/{label}: {gt_values}')
coverage=pd.DataFrame(rows)
coverage.to_csv(OUTPUT_DIR/'coverage_report.csv',index=False)
pd.DataFrame(transformations).to_csv(OUTPUT_DIR/'display_transformations.csv',index=False)
summary={'total_expected':len(pairs)*len(required_sources),'available':int((coverage.status=='ok').sum()),'missing_or_invalid':int((coverage.status!='ok').sum()),'per_source':coverage.groupby('source').status.value_counts().unstack(fill_value=0).to_dict('index'),'errors':issues}
(OUTPUT_DIR/'coverage_summary.json').write_text(json.dumps(summary,indent=2))
display(coverage.groupby(['source','status']).size().unstack(fill_value=0))
if issues: raise RuntimeError(f'{len(issues)} coverage/GT problems. First 30:\n'+'\n'.join(issues[:30])+f'\nSee {OUTPUT_DIR}/coverage_report.csv')


## 5. Ground-truth geometry, faithful overlays, PDF production

The reference uses original-resolution images, boxes scaled by the `test_meta.csv` width/height fields, `magma` coloring, `alpha * heatmap` compositing, lime rectangles, and text labels. Here a GT source is additionally required: at least one stored map's explicit `ground_truth` field for each pair. Disagreements are fatal.


In [ ]:
ann=pd.read_csv(ANNOTATIONS_CSV,dtype={'image_id':str})
meta=pd.read_csv(IMAGE_METADATA_CSV,dtype={'image_id':str})
def col(df,options,meaning):
    lookup={c.casefold():c for c in df.columns}
    for c in options:
        if c.casefold() in lookup:return lookup[c.casefold()]
    raise KeyError(f'Missing {meaning}; available: {list(df.columns)}')
ai=col(ann,['image_id','imageid','image_name'],'annotation image ID')
al=col(ann,['class_name','label','finding_name'],'annotation label')
xy=[col(ann,x,y) for x,y in [(['x_min','xmin','x1'],'xmin'),(['y_min','ymin','y1'],'ymin'),(['x_max','xmax','x2'],'xmax'),(['y_max','ymax','y2'],'ymax')]]
mi=col(meta,['image_id','imageid','image_name'],'metadata ID')
mw=col(meta,['width','original_width','w'],'metadata width')
mh=col(meta,['height','original_height','h'],'metadata height')
ann[ai]=ann[ai].astype(str);meta[mi]=meta[mi].astype(str)
if meta[mi].duplicated().any():raise ValueError('Duplicate image ID in image metadata')
meta_lookup=meta.set_index(mi)[[mw,mh]]

def get_boxes(image_id,label,size):
    if image_id not in meta_lookup.index:raise KeyError(f'Missing size metadata: {image_id}')
    orig=meta_lookup.loc[image_id];sx=size[0]/float(orig[mw]);sy=size[1]/float(orig[mh]);out=[]
    subset=ann[(ann[ai]==image_id)&(ann[al].map(normal_label)==normal_label(label))]
    for r in subset.itertuples(index=False,name=None):
        row=dict(zip(ann.columns,r));x0,y0,x1,y1=[float(row[c]) for c in xy]
        if x1<=x0 or y1<=y0:raise ValueError(f'Invalid GT box {image_id}/{label}: {[x0,y0,x1,y1]}')
        out.append(dict(x=x0*sx,y=y0*sy,w=(x1-x0)*sx,h=(y1-y0)*sy))
    return out

def draw_boxes(ax,boxes,color='lime',lw=1.6):
    for b in boxes:ax.add_patch(Rectangle((b['x'],b['y']),b['w'],b['h'],fill=False,edgecolor=color,linewidth=lw))

def make_overlay(image,vis_map,alpha=OVERLAY_ALPHA):
    rgb=np.asarray(image,dtype=np.float32)/255.0
    heatmap=np.asarray(vis_map,dtype=np.float32)
    if heatmap.shape!=rgb.shape[:2]:raise ValueError(f'Image/heatmap spatial mismatch: {rgb.shape[:2]} vs {heatmap.shape}')
    color=colormaps[CMAP_NAME](np.clip(heatmap,0,1))[...,:3]
    opacity=(alpha*np.clip(heatmap,0,1))[...,None]
    return np.clip((1-opacity)*rgb+opacity*color,0,1)

def gt_status(image_id,label):
    values=[records[(s,image_id,label)]['record'].get('ground_truth') for s in SOURCE_DIRS]
    values=[float(x) for x in values if x is not None and np.isfinite(float(x))]
    if not values:raise ValueError(f'No verified ground truth for {image_id}/{label}; need saved GT or explicitly supplied labels CSV')
    if len(set(values))!=1:raise ValueError(f'Conflicting ground truth for {image_id}/{label}: {values}')
    return int(values[0]) if values[0] in (0,1) else float(values[0])

def stat_title(title,rec):
    # Original fusion convention p/m only if actually recorded; never fabricate scores.
    vals=[]
    for name,field in [('p','positive_probability'),('m','margin')]:
        v=rec.get(field)
        if isinstance(v,(int,float)) and np.isfinite(v): vals.append(f'{name}={v:.4f}')
    return title + ('\n'+' | '.join(vals) if vals else '')

def render_page(page_pairs,page_number,start,spec,head_title):
    n=len(page_pairs);ncols=1+len(spec)
    # Keep 3.6 inches/row and ~3.43 inches/column, matching the 24x3.6n reference density.
    fig,axes=plt.subplots(n,ncols,figsize=(24*ncols/7,3.6*n),dpi=FIG_DPI,squeeze=False)
    for r,item in enumerate(page_pairs.itertuples(index=False)):
        with Image.open(IMAGES_ROOT/f'{item.image_id}.png') as im:image=im.convert('RGB')
        boxes=get_boxes(item.image_id,item.label,image.size)
        axes[r,0].imshow(image);draw_boxes(axes[r,0],boxes)
        axes[r,0].set_title(f'{start+r+1:03d}. {item.image_id}\n{item.label} | GT={gt_status(item.image_id,item.label)}',fontsize=8)
        for c,(source,title) in enumerate(spec,start=1):
            entry=records[(source,item.image_id,item.label)]
            axes[r,c].imshow(make_overlay(image,entry['map']));draw_boxes(axes[r,c],boxes)
            axes[r,c].set_title(stat_title(title,entry['record']),fontsize=8)
        for ax in axes[r]:ax.axis('off')
    fig.suptitle(f'{head_title} - page {page_number:02d}',fontsize=14,fontweight='bold',y=1.002)
    plt.tight_layout()
    return fig

# Verify metadata and labels for ALL 100 pairs before writing any PDF.
for item in pairs.itertuples(index=False):
    with Image.open(IMAGES_ROOT/f'{item.image_id}.png') as im: get_boxes(item.image_id,item.label,im.size)
    gt_status(item.image_id,item.label)

for filename,spec in PDF_SPECS:
    path=OUTPUT_DIR/filename
    title=' | '.join(name.split(' - ')[0] for _,name in spec)
    with PdfPages(path) as pdf:
        for page_index in range(math.ceil(len(pairs)/CASES_PER_PAGE)):
            start=page_index*CASES_PER_PAGE
            fig=render_page(pairs.iloc[start:start+CASES_PER_PAGE],page_index+1,start,spec,title)
            pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    print('Saved',path)


## 6. Optional compact main-thesis figure (6–8 **predeclared** cases)

To prevent cherry-picking, supply a fixed set of 6–8 `(image_id, pathology)` pairs from the canonical analysis *before* reviewing comparative heatmaps. This adds a separate PDF and never changes the 100-case analysis. Use `None` to skip.


In [ ]:
THESIS_CASE_PAIRS = None # e.g. [('exact_id_from_manifest','Cardiomegaly'), ...] — YOU choose and freeze
THESIS_COMPARISON_INDEX = 2 # 0=anatomy strategies, 1=lambda ablation, 2=A5
if THESIS_CASE_PAIRS is not None:
    if not 6<=len(THESIS_CASE_PAIRS)<=8:raise ValueError('Choose 6–8 unique representative pairs')
    if len(set(THESIS_CASE_PAIRS))!=len(THESIS_CASE_PAIRS):raise ValueError('Duplicate representative cases')
    available=set(map(tuple,pairs[['image_id','label']].to_records(index=False)))
    if any(tuple(x) not in available for x in THESIS_CASE_PAIRS):raise ValueError('Thesis figure may only use canonical cases')
    order={tuple(x):i for i,x in enumerate(THESIS_CASE_PAIRS)}
    selected=pairs[pairs.apply(lambda row:(row.image_id,row.label) in order,axis=1)].copy()
    selected['_order']=selected.apply(lambda row:order[(row.image_id,row.label)],axis=1)
    selected=selected.sort_values('_order',kind='stable')
    spec=PDF_SPECS[THESIS_COMPARISON_INDEX][1]
    with PdfPages(OUTPUT_DIR/'thesis_representative_cases.pdf') as pdf:
        for k in range(0,len(selected),CASES_PER_PAGE):
            fig=render_page(selected.iloc[k:k+CASES_PER_PAGE],k//CASES_PER_PAGE+1,k,spec,'Thesis examples')
            pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    print('Saved thesis_representative_cases.pdf')
else:
    print('Skipped optional thesis figure; THESIS_CASE_PAIRS not provided.')


## Methodology and interpretation constraints

- **Original ALBEF:** The uploaded margin extraction script defines ITC attention Grad-CAM using gradients of `(positive cosine similarity - negative cosine similarity) / temperature` with respect to visual self-attention, keeping the positive part of attention × gradient and max-normalizing for display. The actual selected baseline must be checked against its per-file metadata.
- **A1–A4:** The experiment names describe training objectives, **not independently established extraction mechanisms**. Confirm each saved map's `method`, `target`, `vit_layer_index`, prompt scheme, normalization, and source checkpoint from its real payload/manifest. If comparable ITC margin Grad-CAM was applied to each variant, report that only once independently verified.
- **A5 raw:** Pathology-specific cross-attention used as patch-head input; **not automatically ITC Grad-CAM**. Verify attention aggregation/head/token conventions from the actual A5 extractor and saved payload.
- **A5 learned:** Learned patch-level distribution, **not Grad-CAM**. Preserve the saved spatial distribution or documented visualization representation; if saved as probability mass rather than a display-normalized intensity map, set an explicitly justified display transform instead of per-case normalization that hides scale.
- **Transformations recorded:** `display_transformations.csv` lists any bilinear alignment resizing. The only other allowed changes are the declared fixed-range transform (if used) and display clipping to [0,1], the latter matching the reference notebook. No per-image min–max rescaling is performed here.
- **Read-only heatmap policy:** This notebook never imports model architecture or inference functions and never writes or alters original saved heatmaps.
- **Coverage:** All 100 canonical pairs × all unique required sources must be validated, including negative cases; missing/corrupted entries are reported, then plotting stops.
- **Qualitative only:** No FROC, IoU, Dice or other localization metrics are computed.
